In [ ]:
# Per-class analysis of retrieval evaluation results using individual CLIP scores
# Compare trimodal vs bimodal model performance at the sample level
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np
from collections import defaultdict
import re
print('Starting per-class retrieval analysis...')

In [ ]:
# Load individual CLIP scores from retrieval evaluations
results = []

# Parse input files to extract model type and test dataset
for result_file in snakemake.input:
    df = pd.read_csv(result_file)
    
    # Extract model type and dataset from file path
    path_parts = Path(result_file).parts
    
    # Find the log directory name that contains model and dataset info
    log_dir = None
    for part in path_parts:
        if part.startswith('sweval___'):
            log_dir = part
            break
    
    if log_dir:
        # Parse: sweval___{model}___{test_dataset}
        parts = log_dir.split('___')
        if len(parts) >= 3:
            model_name = parts[1]
            test_dataset = parts[2]
            
            # Map to model type using the provided mappings
            model_type = 'unknown'
            for mtype, mname in snakemake.params.model_mappings.items():
                if model_name == mname:
                    model_type = mtype
                    break
            
            # Add metadata to scores
            df['model_type'] = model_type
            df['model_name'] = model_name
            df['test_dataset'] = test_dataset
            df['result_file'] = str(result_file)
            
            results.append(df)

combined_df = pd.concat(results, ignore_index=True)
print(f'Loaded {len(combined_df)} CLIP score entries from {len(results)} files')
print(combined_df.head())

In [ ]:
# Create comparison between trimodal and bimodal models using CLIP scores
model_comparison = pd.DataFrame()

# Use average of both directions as overall CLIP score
combined_df['avg_clip_score'] = (combined_df['clip_score_left_right'] + combined_df['clip_score_right_left']) / 2

# Group by sample and compute statistics
sample_comparison = combined_df.groupby(['test_dataset', 'orig_ids', 'model_type'])['avg_clip_score'].mean().reset_index()

# Pivot to get model types as columns
pivot_df = sample_comparison.pivot_table(
    index=['test_dataset', 'orig_ids'],
    columns='model_type',
    values='avg_clip_score',
    aggfunc='mean'
).reset_index()

# Calculate improvement (trimodal - bimodal_matching)
if 'trimodal' in pivot_df.columns and 'bimodal_matching' in pivot_df.columns:
    pivot_df['improvement'] = pivot_df['trimodal'] - pivot_df['bimodal_matching']
    pivot_df['relative_improvement'] = (
        pivot_df['improvement'] / pivot_df['bimodal_matching'] * 100
    )
    
    model_comparison = pivot_df
    print('Model comparison created:')
    print(model_comparison.head(10))
else:
    print('Required model types not found in data')
    print('Available model types:', sample_comparison['model_type'].unique())
    print('No data available for comparison')

In [ ]:
# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(20, 16))
fig.suptitle(f'Retrieval Per-Sample Performance: Trimodal vs Bimodal Models\nTest Dataset: {snakemake.wildcards.dataset}', fontsize=16)

# Plot 1: Top 10 improving samples
top_improving = model_comparison.nlargest(10, 'improvement')
y_pos = np.arange(len(top_improving))
colors = ['green' if x >= 0 else 'red' for x in top_improving['improvement']]
axes[0,0].barh(y_pos, top_improving['improvement'], color=colors, alpha=0.7)
axes[0,0].set_yticks(y_pos)
axes[0,0].set_yticklabels([f"Sample {row['orig_ids']}" for _, row in top_improving.iterrows()])
axes[0,0].set_xlabel('CLIP Score Improvement')
axes[0,0].set_title('Top 10 Samples: CLIP Score Improvement\n(Trimodal - Bimodal)')
axes[0,0].axvline(x=0, color='black', linestyle='--', alpha=0.5)
axes[0,0].grid(True, alpha=0.3)

# Plot 2: Bottom 10 samples (most declining)
bottom_samples = model_comparison.nsmallest(10, 'improvement')
y_pos = np.arange(len(bottom_samples))
colors = ['red' if x < 0 else 'green' for x in bottom_samples['improvement']]
axes[0,1].barh(y_pos, bottom_samples['improvement'], color=colors, alpha=0.7)
axes[0,1].set_yticks(y_pos)
axes[0,1].set_yticklabels([f"Sample {row['orig_ids']}" for _, row in bottom_samples.iterrows()])
axes[0,1].set_xlabel('CLIP Score Change')
axes[0,1].set_title('Bottom 10 Samples: CLIP Score Change\n(Trimodal - Bimodal)')
axes[0,1].axvline(x=0, color='black', linestyle='--', alpha=0.5)
axes[0,1].grid(True, alpha=0.3)

# Plot 3: Scatter plot of trimodal vs bimodal scores
axes[1,0].scatter(model_comparison['bimodal_matching'], model_comparison['trimodal'], alpha=0.6)
# Add diagonal line
min_val = min(model_comparison['bimodal_matching'].min(), model_comparison['trimodal'].min())
max_val = max(model_comparison['bimodal_matching'].max(), model_comparison['trimodal'].max())
axes[1,0].plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.8, label='Equal Performance')
axes[1,0].set_xlabel('Bimodal CLIP Score')
axes[1,0].set_ylabel('Trimodal CLIP Score')
axes[1,0].set_title('Sample-wise Performance Comparison')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# Plot 4: Distribution of improvements
axes[1,1].hist(model_comparison['improvement'], bins=20, alpha=0.7, color='blue')
axes[1,1].axvline(x=0, color='red', linestyle='--', alpha=0.8, label='No Change')
axes[1,1].axvline(x=model_comparison['improvement'].mean(), color='green', 
                 linestyle='-', alpha=0.8, label='Mean')
axes[1,1].set_xlabel('CLIP Score Improvement')
axes[1,1].set_ylabel('Number of Samples')
axes[1,1].set_title('Distribution of CLIP Score Improvements')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

fig.savefig(snakemake.output.plot)

In [ ]:
# Statistical analysis of improvements
print(f'=== RETRIEVAL STATISTICAL SUMMARY ({snakemake.wildcards.dataset}) ===')

improvements = model_comparison['improvement'].dropna()

print(f'\nCLIP Score Improvements:')
print(f'Mean improvement: {improvements.mean():.4f}')
print(f'Median improvement: {improvements.median():.4f}')
print(f'Std deviation: {improvements.std():.4f}')
print(f'Samples with positive improvement: {(improvements > 0).sum()}/{len(improvements)}')
print(f'Max improvement: {improvements.max():.4f}')
print(f'Min improvement: {improvements.min():.4f}')

# Top and bottom samples
print(f'\nTop 5 improving samples:')
top_5 = model_comparison.nlargest(5, 'improvement')
for _, row in top_5.iterrows():
    print(f'  Sample {row["orig_ids"]}: +{row["improvement"]:.4f} (trimodal: {row["trimodal"]:.4f}, bimodal: {row["bimodal_matching"]:.4f})')

print(f'\nBottom 5 samples:')
bottom_5 = model_comparison.nsmallest(5, 'improvement')
for _, row in bottom_5.iterrows():
    print(f'  Sample {row["orig_ids"]}: {row["improvement"]:.4f} (trimodal: {row["trimodal"]:.4f}, bimodal: {row["bimodal_matching"]:.4f})')

# Save detailed results
model_comparison.to_csv(snakemake.output.analysis, index=False)
print(f'\nDetailed results saved to: {snakemake.output.analysis}')